# POLG case-control analysis 
- **Project**: Multi-ancestry analysis of POLG variants in Parkinson’s disease
- **Last Update:** APRIL-2026

## Getting started

### Loading Python libraries

In [ ]:
# Use the os package to interact with the environment
import os

# Bring in Pandas for Dataframe functionality
import pandas as pd

import subprocess

# Numpy for basics
import numpy as np

# Use pathlib for file path manipulation
import pathlib

# Use StringIO for working with file contents
from io import StringIO

# Enable IPython to display matplotlib graphs
import matplotlib.pyplot as plt
%matplotlib inline

# Import the iPython HTML rendering for displaying links to Google Cloud Console
from IPython.core.display import display, HTML

# Import urllib modules for building URLs to Google Cloud Console
import urllib.parse

# BigQuery for querying data
from google.cloud import bigquery

#Import Sys
import sys as sys

import re

### Set paths

In [ ]:
#To access data of GP2 version 11.0
REL11_PATH = pathlib.Path(pathlib.Path.home(), '/path/to/gp2/release11')
!ls -hal {REL11_PATH}/clinical_data

In [ ]:
#To set path to each bucket
EXTENDED_CLINICAL_DATA_PATH = pathlib.Path(REL11_PATH, 'clinical_data/r11_extended_clinical_data.csv')
CLINICAL_DATA_PATH          = pathlib.Path(REL11_PATH, 'clinical_data/master_key_release11_final_vwb.csv')
EXTENDED_CLINICAL_DATA_PATH = pathlib.Path(REL11_PATH, 'clinical_data/r11_extended_clinical_data_vwb.csv')
RELATED_DATA_PATH           = pathlib.Path(REL11_PATH, 'meta_data/related_samples/')
RAW_GENO_PATH               = pathlib.Path(REL11_PATH, 'raw_genotypes')
IMPUTED_GENO_PATH           = pathlib.Path(REL11_PATH, 'imputed_genotypes')
PCS_PATH                    = pathlib.Path(REL11_PATH, 'imputed_genotypes')

### Install Packages

In [ ]:
%%capture
%%bash

#To install plink 1.9
cd /home/jupyter/
if test -e /home/jupyter/plink; then
    echo "Plink is already installed in /home/jupyter/"
else
    echo "Plink is not installed"
    cd /home/jupyter

    wget http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip 

    unzip -o plink_linux_x86_64_20190304.zip
    mv plink plink1.9
fi

In [ ]:
%%bash

#chmod plink 1.9 to ensure permission to run the program
chmod u+x /home/jupyter/plink1.9

In [ ]:
%%capture
%%bash

#To install plink 2.0
cd /home/jupyter/
if test -e /home/jupyter/plink2; then

echo "Plink2 is already installed in /home/jupyter/"
else
echo "Plink2 is not installed"
cd /home/jupyter/

wget http://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip

unzip -o plink2_linux_x86_64_latest.zip

fi

In [ ]:
%%bash

#chmod plink 2 to ensure permission to run the program
chmod u+x /home/jupyter/plink2

In [ ]:
%%bash

#To install ANNOVAR after registration on the annovar website - https://www.openbioinformatics.org/annovar/annovar_download_form.php

if test -e /home/jupyter/annovar; then

echo "annovar is already installed in /home/jupyter/"
else
echo "annovar is not installed"
cd /home/jupyter/

wget http://www.openbioinformatics.org/annovar/download/0wgxR2rIVP/annovar.latest.tar.gz

tar xvfz annovar.latest.tar.gz

fi

In [ ]:
!wget http://www.openbioinformatics.org/annovar/download/hg38_clinvar_20240908.txt.gz

In [ ]:
%%bash

#To download resources for annotation

cd /home/jupyter/annovar/

# Commented out to avoid downloading all at the same time
# Uncomment lines to download these resources
# perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar clinvar_20250721 humandb/
# perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar dbnsfp47a humandb/ 
# perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar refGene humandb/
# perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar gnomad41_genome humandb/
# perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar gnomad41_exome humandb/

## Create a covariate file for GP2 data

In [ ]:
#To load master key 
key = pd.read_csv(CLINICAL_DATA_PATH, low_memory=False)
print(f'Clinical data (num rows, num columns): {key.shape}')
pd.set_option('display.max_columns', None)
key.head()

In [ ]:
#To subset master key to keep only a few columns 
key = key[['GP2ID', 'baseline_GP2_phenotype_for_qc', 'biological_sex_for_qc', 'age_at_sample_collection', 'age_of_onset', 'nba_label','wgs_label']]
# Renaming the columns
key.rename(columns = {'GP2ID':'IID',
                                     'baseline_GP2_phenotype_for_qc':'phenotype',
                                     'biological_sex_for_qc':'SEX', 
                                     'age_at_sample_collection':'AGE', 
                                     'age_of_onset':'AAO'}, inplace = True)

In [ ]:
#To tidy ancestry label for NBA and WGS samples
key["label"] = key["nba_label"].combine_first(key["wgs_label"])
key = key.drop(columns=["nba_label", "wgs_label"])
#key

In [ ]:
#To make directories for each ancestry
ancestries = {'AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH'}

for ancestry in ancestries:
    !mkdir {WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}

In [ ]:
#To tidy and create demographic subfiles from master key for each ancestry 
ancestries = {'AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH'}

for ancestry in ancestries:
    
    print(f'WORKING ON: {ancestry}')
    
    ## Subset to keep ancestry of interest 
    ancestry_key = key[key['label']==ancestry].copy()
    ancestry_key.reset_index(drop=True)
    
    # Convert phenotype to binary (1/2)
    ## Assign conditions so case=2 and controls=1, and -9 otherwise (matching PLINK convention)
    # PD = 2; control = 1
    pheno_mapping = {"PD": 2, "Control": 1}
    ancestry_key['PHENO'] = ancestry_key['phenotype'].map(pheno_mapping).astype('Int64')

    # Check value counts of pheno
    ancestry_key['PHENO'].value_counts(dropna=False)
    
    ## Get the PCs
    pcs = pd.read_csv(f'{RAW_GENO_PATH}/{ancestry}/{ancestry}_release11_vwb.eigenvec', sep='\t')
    
    #Select just first 5 PCs
    selected_columns = ['IID', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
    pcs = pd.DataFrame(data=pcs.iloc[:, 1:7].values, columns=selected_columns)

    # Drop the first row (since it's now the column names)
    pcs = pcs.drop(0)

    # Reset the index to remove any potential issues
    pcs = pcs.reset_index(drop=True)
    
    # Check size
    print(f'PCs: {pcs.shape}')

    # Check value counts of SEX
    sex_og_values = ancestry_key['SEX'].value_counts(dropna=False)
    print(f'Sex value counts - original:\n {sex_og_values.to_string()}')
    
    # Convert sex to binary (1/2)
    ## Assign conditions so female=2 and men=1, and -9 otherwise (matching PLINK convention)
    # Female = 2; Male = 1
    sex_mapping = {"Female": 2, "Male": 1}
    ancestry_key['SEX'] = ancestry_key['SEX'].map(sex_mapping).astype('Int64')
    
    # Check value counts of SEX after recoding
    sex_recode_values = ancestry_key['SEX'].value_counts(dropna=False)
    print(f'Sex value counts - recoded:\n{sex_recode_values.to_string()}')
    
    ## Make covariate file
    df = pd.merge(ancestry_key,pcs, on='IID')
    print(f'Check columns for covariate file: {df.columns}')

    # Load information about related individuals in the ancestry analyzed
    related_df = pd.read_csv(f'{RELATED_DATA_PATH}/{ancestry}_release11_vwb.related')
    print(f'Related individuals: {related_df.shape}')
    
    # Make a list of just one set of related people
    related_list = list(related_df['IID1'])
    related_list = [re.sub(r'_s.*$', '', iid) for iid in related_list]
    print(f'Number of related IIDs in dataset before filtering: {df["IID"].isin(related_list).sum()}')
    
    # Check value counts of related and remove only one related individual
    df = df[~df["IID"].isin(related_list)]

    # Check size
    print(f'Unrelated individuals: {df.shape}')
    
    #Make additional columns - FID, fatid and matid - these are needed for RVtests!!
    #RVtests needs the first 5 columns to be fid, iid, fatid, matid and sex otherwise it does not run correctly
    #Uppercase column name is ok
    #See https://zhanxw.github.io/rvtests/#phenotype-file
    df['FID'] = 0
    df['FATID'] = 0
    df['MATID'] = 0

    ## Clean up and keep columns we need 
    final_df = df[['FID','IID', 'FATID', 'MATID', 'SEX', 'AGE','AAO', 'PHENO','PC1', 'PC2', 'PC3', 'PC4', 'PC5']].copy()

    ##DO NOT replace missing values with -9 as this is misinterpreted by RVtests - needs to be nonnumeric
    #Leave missing values as NA
    
    #Check number of PD cases missing age
    pd_missAge = final_df[(final_df['PHENO']==2)&(final_df['AGE'].isna())]
    print(f'Number of PD cases missing age: {pd_missAge.shape[0]}')
    
    #Check number of controls missing age
    control_missAge = final_df[(final_df['PHENO']==1)&(final_df['AGE'].isna())]
    print(f'Number of controls missing age: {control_missAge.shape[0]}')

    ## Make file of sample IDs to keep 
    samples_toKeep = final_df[['FID', 'IID']].copy()
    samples_toKeep.columns = ['#FID','IID']
    
    samplestokeep_path = pathlib.Path(pathlib.Path.home(), f'{WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}.samplestokeep')
    
    # Create the output CSV file's parent folder in the cloud storage bucket, if it doesn't already exist.
    if not samplestokeep_path.parent.exists():
        !mkdir -p {samplestokeep_path.parent}
        print(f'Created {samplestokeep_path.parent}')
    
    samples_toKeep.to_csv(samplestokeep_path, sep = '\t', index=False)

    finaldf_path = pathlib.Path(pathlib.Path.home(), f'{WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_covariate_file.txt')
    
    # Create the output CSV file's parent folder in the cloud storage bucket, if it doesn't already exist.
    if not finaldf_path.parent.exists():
        !mkdir -p {finaldf_path.parent}
        print(f'Created {finaldf_path.parent}')
    
    final_df.to_csv(finaldf_path, sep = '\t', na_rep='NA', index=False)

## Annotation of the gene

### Extract region of interest using PLINK

- Extract *POLG* gene
- *POLG* coordinates: Chromosome 15: 89,305,198-89,334,861 (GRCh38/hg38)

In [ ]:
#To extract POLG region using plink
for ancestry in ancestries:
    
    ! /home/jupyter/plink2 \
    --pfile {IMPUTED_GENO_PATH}/{ancestry}/chr15_{ancestry}_release11_vwb \
    --chr 15 \
    --from-bp 89305198 \
    --to-bp 89334861 \
    --mac 2 \
    --hwe 1e-5 0.001 \
    --make-pgen \
    --out {WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG

### Turn binary files into VCF

In [ ]:
ancestries = {'AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH'}
for ancestry in ancestries:
        
    ## Turn binary files into VCF
    ! /home/jupyter/plink2 \
    --pfile {WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG \
    --recode vcf id-paste=iid \
    --out {WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG

In [ ]:
!sudo apt-get update
!sudo apt-get install tabix

In [ ]:
#Bgzip and Tabix - to zip and index the VCF files
ancestries = {'AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH'}
for ancestry in ancestries:    
    ! bgzip -f {WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG.vcf
    ! tabix -f -p vcf {WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG.vcf.gz 

### Annotate using ANNOVAR

In [ ]:
## annotate using ANNOVAR
ancestries = {'AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH'}

for ancestry in ancestries:
        
    ! perl /home/jupyter/annovar/table_annovar.pl {WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG.vcf.gz /home/jupyter/annovar/humandb/ -buildver hg38 \
    -out {WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG.annovar \
    -remove -protocol refGene,clinvar_20250721,dbnsfp47a,gnomad41_genome,gnomad41_exome \
    -operation g,f,f,f,f \
    --nopolish \
    -nastring . \
    -vcfinput

In [ ]:
#Overview of variants identified and grouping of variants into different subgroups
import pandas as pd

ancestries = ['AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH']

for ancestry in ancestries:
    print(f'WORKING ON: {ancestry}')
    
    # Read in ANNOVAR multianno file
    gene = pd.read_csv(f'{WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG.annovar.hg38_multianno.txt', sep='\t')
    
    # Filter for gene POLG only
    gene = gene[gene['Gene.refGene'] == 'POLG']
    
    # Split clinvar column
    #gene_split = gene['clinvar_20250721'].str.split(';', expand=True)
    #gene_split.columns = [f'clinvar_part_{i+1}' for i in range(gene_split.shape[1])]
    #gene = pd.concat([gene, gene_split], axis=1)
    #gene['CLINSIG'] = gene['clinvar_part_1']
    
    # Convert columns to numeric safely
    for col in ['gnomad41_genome_fafmax_faf95_max', 'CADD_phred', 'REVEL_score','PrimateAI_score']:
        gene[col] = pd.to_numeric(gene[col], errors='coerce')

    # Define the complex pathogenicity condition once
    pathogenicity_condition = (
    ((gene['gnomad41_genome_fafmax_faf95_max'] < 0.01) &
     (gene['CADD_phred'] > 20) &
     (
        (gene['REVEL_score'] > 0.644) |
        (gene['Polyphen2_HDIV_pred'] == 'D') |
        (gene['SIFT4G_pred'] == 'D') |
        (gene['PrimateAI_score'] > 0.8)
     )
    )
    |
    gene['CLNSIG'].str.contains(r'\blikely pathogenic\b|\bpathogenic\b', case=False, na=False, regex=True))

    # Apply filters using conditions
    filtered_variants = gene[gene['gnomad41_genome_fafmax_faf95_max'] < 0.03]
    filtered_variants2 = gene[gene['gnomad41_genome_fafmax_faf95_max'] < 0.01]
    filtered_variants3 = gene[pathogenicity_condition]
    filtered_variants4 = gene[pathogenicity_condition & (gene['Func.refGene'] == 'exonic')]
    filtered_variants5 = gene[pathogenicity_condition & (gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'nonsynonymous SNV')]
    
    # Now you can calculate variant counts by categories on filtered_variants
    intronic = filtered_variants[filtered_variants['Func.refGene'] == 'intronic']
    upstream = filtered_variants[filtered_variants['Func.refGene'] == 'upstream']
    downstream = filtered_variants[filtered_variants['Func.refGene'] == 'downstream']
    utr5 = filtered_variants[filtered_variants['Func.refGene'] == 'UTR5']
    utr3 = filtered_variants[filtered_variants['Func.refGene'] == 'UTR3']
    splicing = filtered_variants[filtered_variants['Func.refGene'] == 'splicing']
    exonic = filtered_variants[filtered_variants['Func.refGene'] == 'exonic']
    stopgain = filtered_variants[(filtered_variants['Func.refGene'] == 'exonic') & (filtered_variants['ExonicFunc.refGene'] == 'stopgain')]
    stoploss = filtered_variants[(filtered_variants['Func.refGene'] == 'exonic') & (filtered_variants['ExonicFunc.refGene'] == 'stoploss')]
    startloss = filtered_variants[(filtered_variants['Func.refGene'] == 'exonic') & (filtered_variants['ExonicFunc.refGene'] == 'startloss')]
    frameshift_deletion = filtered_variants[(filtered_variants['Func.refGene'] == 'exonic') & (filtered_variants['ExonicFunc.refGene'] == 'frameshift deletion')]
    frameshift_insertion = filtered_variants[(filtered_variants['Func.refGene'] == 'exonic') & (filtered_variants['ExonicFunc.refGene'] == 'frameshift insertion')]
    nonframeshift_deletion = filtered_variants[(filtered_variants['Func.refGene'] == 'exonic') & (filtered_variants['ExonicFunc.refGene'] == 'nonframeshift deletion')]
    nonframeshift_insertion = filtered_variants[(filtered_variants['Func.refGene'] == 'exonic') & (filtered_variants['ExonicFunc.refGene'] == 'nonframeshift insertion')]
    coding_nonsynonymous = filtered_variants[(filtered_variants['Func.refGene'] == 'exonic') & (filtered_variants['ExonicFunc.refGene'] == 'nonsynonymous SNV')]
    coding_synonymous = filtered_variants[(filtered_variants['Func.refGene'] == 'exonic') & (filtered_variants['ExonicFunc.refGene'] == 'synonymous SNV')]
    
    print(f"{ancestry}")
    print('Total filtered variants: ', len(filtered_variants))
    print("Intronic: ", len(intronic))
    print("Upstream: ", len(upstream))
    print("Downstream: ", len(downstream))
    print('UTR3: ', len(utr3))
    print('UTR5: ', len(utr5))
    print("Splicing: ", len(splicing))
    print("Total exonic: ", len(exonic))
    print("Stopgain: ", len(stopgain))
    print("Stoploss: ", len(stoploss))
    print("Startloss: ", len(startloss))
    print("Frameshift deletion: ", len(frameshift_deletion))
    print("Frameshift insertion: ", len(frameshift_insertion))
    print("Non-frameshift insertion: ", len(nonframeshift_insertion))
    print("Non-frameshift deletion: ", len(nonframeshift_deletion))
    print('Synonymous: ', len(coding_synonymous))
    print("Nonsynonymous: ", len(coding_nonsynonymous))
    print('\n')
    
    ## For rvtests
    
    # Potential functional: These are variants annotated as frameshift, nonframeshift, startloss, stoploss, stopgain, splicing, missense, exonic, UTR5, UTR3, upstream (-100bp), downstream (+100bp), or ncRNA. 
    potentially_functional = gene[gene['Func.refGene'] != 'intronic']
    # Coding: These are variants annotated as frameshift, nonframeshift, startloss, stoploss, stopgain, splicing, or missense.
    coding_variants = gene[(gene['Func.refGene'] == 'splicing') | (gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] != 'synonymous SNV')]
    # Loss of function: These are variants annotated as frameshift, startloss,stopgain, or splicing.
    loss_of_function = gene[(gene['Func.refGene'] == 'splicing') | (gene['ExonicFunc.refGene'] == 'stopgain') | (gene['ExonicFunc.refGene'] == 'startloss') | (gene['ExonicFunc.refGene'] == 'frameshift deletion') | (gene['ExonicFunc.refGene'] == 'frameshift insertion')]
    
    # Save in PLINK format
    variants_toKeep = filtered_variants[['Chr','Start','End','Gene.refGene']].copy()
    variants_toKeep.to_csv(f'{WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG.MAF3.variantstoKeep.txt', sep="\t", index=False, header=False)

    variants_toKeep2 = filtered_variants2[['Chr','Start','End','Gene.refGene']].copy()
    variants_toKeep2.to_csv(f'{WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG.MAF1.variantstoKeep.txt', sep="\t", index=False, header=False)

    variants_toKeep3 = filtered_variants3[['Chr','Start','End','Gene.refGene']].copy()
    variants_toKeep3.to_csv(f'{WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG.pathogenic.variantstoKeep.txt', sep="\t", index=False, header=False)

    variants_toKeep4 = filtered_variants4[['Chr','Start','End','Gene.refGene']].copy()
    variants_toKeep4.to_csv(f'{WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG.pathogenic_coding.variantstoKeep.txt', sep="\t", index=False, header=False)

    variants_toKeep5 = filtered_variants5[['Chr','Start','End','Gene.refGene']].copy()
    variants_toKeep5.to_csv(f'{WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG.pathogenic_missense.variantstoKeep.txt', sep="\t", index=False, header=False)
    
    # For assoc
    
    # These are all exonic variants
    exonic = gene[gene['Func.refGene'] == 'exonic']
    
    # Save in PLINK format
    variants_toKeep6 = exonic[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
    variants_toKeep6.to_csv(f'{WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG.exonic.variantstoKeep.txt', sep="\t", index=False, header=False)

## Case/Control Analysis

### Glossary

- CHR Chromosome code
- SNP Variant identifier
- A1 Allele 1 (usually minor)
- A2 Allele 2 (usually major)
- MAF Allele 1 frequency in all subjects
- F_A/MAF_A Allele 1 frequency in cases
- F_U/MAF_U Allele 1 frequency in controls
- NCHROBS_A Number of case allele observations
- NCHROBS_U Number of control allele observations

### glm

In [ ]:
%%bash
ancestries=(AAC AFR AJ AMR CAS EAS EUR FIN MDE SAS CAH)

for ancestry in "${ancestries[@]}"; do
  file="{WORK_DIR}/POLG_GP2_NBA_R11/${ancestry}/${ancestry}_POLG.psam"

  echo "Updating $file ..."

  awk 'BEGIN{OFS="\t"}
       NR==1 {
         for (i=1; i<=NF; i++) {
           if ($i=="#FID") fid=i
         }
         print
         next
       }
       {
         $fid=0
         print
       }' "$file" > "${file}.tmp" && mv "${file}.tmp" "$file"

done

In [ ]:
%%bash
ancestries=(AAC AFR AJ AMR CAS EAS EUR FIN MDE SAS CAH)

for ancestry in "${ancestries[@]}"; do
  file="{WORK_DIR}/POLG_GP2_NBA_R11/${ancestry}/${ancestry}_POLG.fam"

  echo "Updating $file ..."

  awk 'BEGIN{OFS=" "} { $1=0; print }' "$file" > "${file}.tmp" && mv "${file}.tmp" "$file"
done


In [ ]:
#Run case-control analysis for all variants with covariates
ancestries = ['AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH']

for ancestry in ancestries:
    
    ! /home/jupyter/plink2 \
    --pfile {WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG \
    --keep {WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}.samplestokeep \
    --glm \
    --extract-if-info 'R2>=0.8' \
    --ci 0.95 \
    --adjust \
    --maf 0.01 \
    --mac 2 \
    --hwe 1e-5 0.001 \
    --covar {WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_covariate_file.txt \
    --covar-name SEX,AGE,PC1,PC2,PC3,PC4,PC5 \
    --covar-variance-standardize \
    --out {WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG.allvariants
    
    #--recode A creates a new text fileset, showing each variant in each case and control for the minor allele (A). 
    ! /home/jupyter/plink1.9 \
    --bfile {WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG \
    --keep {WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}.samplestokeep \
    --recode A \
    --out {WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG.allvariants

In [ ]:
#To process results from plink glm analysis for ALL variants

ancestries = ['AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH']

for ancestry in ancestries:
    
    print(f'WORKING ON: {ancestry}')
    
    #Read in glm results
    assoc = pd.read_csv(f'{WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG.allvariants.PHENO1.glm.logistic.hybrid', delim_whitespace=True)
    assoc_add = assoc[assoc['TEST']=="ADD"]
    
    #Filter for significant variants p < 0.05 - if any
    significant = assoc_add[assoc_add['P']<0.05]
    print(f'There are {len(significant)} variants with p-value < 0.05 in glm')
    
    #Read in plink recoded data (.raw file)
    recode = pd.read_csv(f'{WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG.allvariants.raw', delim_whitespace=True)

    # Make a list from the column names
    column_names = recode.columns.tolist()

    # Drop the first 6 columns to keep the variants 
    variants = column_names[6:]

    print(f'Number of variants in {ancestry} for POLG: {len(variants)}')

    # Pre-filter the dataset
    cases_data = recode[recode['PHENOTYPE'] == 2]
    controls_data = recode[recode['PHENOTYPE'] == 1]

    results = []

    # Pre-filter the dataset
    total_cases = cases_data.shape[0]
    total_controls = controls_data.shape[0]
    results = []

    for variant in variants:
        ## For PD cases
        hom_cases = (cases_data[variant] == 2).sum()
        het_cases = (cases_data[variant] == 1).sum()
        hom_ref_cases = (cases_data[variant] == 0).sum()  # Homozygous reference genotype
        missing_cases = total_cases - (hom_cases + het_cases + hom_ref_cases)  # Missing data count
        freq_cases = (2 * hom_cases + het_cases) / (2 * (total_cases - missing_cases))  # Adjust for missing data in denominator

        ## For controls
        hom_controls = (controls_data[variant] == 2).sum()
        het_controls = (controls_data[variant] == 1).sum()
        hom_ref_controls = (controls_data[variant] == 0).sum()  # Homozygous reference genotype
        missing_controls = total_controls - (hom_controls + het_controls + hom_ref_controls)  # Missing data count
        freq_controls = (2 * hom_controls + het_controls) / (2 * (total_controls - missing_controls))  # Adjust for missing data in denominator
    
        # Append results in dictionary format
        results.append({
            'Variant': variant,
            'Hom Cases': hom_cases,
            'Het Cases': het_cases,
            'Hom Ref Cases': hom_ref_cases,
            'Missing Cases': missing_cases,
            'Total Cases': total_cases,
            'Carrier Freq in Cases': freq_cases,
            'Hom Controls': hom_controls,
            'Het Controls': het_controls,
            'Hom Ref Controls': hom_ref_controls,
            'Missing Controls': missing_controls,
            'Total Controls': total_controls,
            'Carrier Freq in Controls': freq_controls
        })

    # Return
    df_results = pd.DataFrame(results)
    df_results['ID'] = df_results['Variant'].apply(lambda x: x.rsplit('_', 1)[0])

    #Print dimensions of the df_results dataframe
    print(f'df_results shape: {df_results.shape}')
    
    #Merge with the glm file
    sig_merge = assoc_add[['ID','A1','A1_FREQ','OBS_CT','OR','LOG(OR)_SE','L95', 'U95','Z_STAT','P']]
    merged = pd.merge(df_results, sig_merge, on='ID', how='right')
    
    bonf = pd.read_csv(f'{WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG.allvariants.PHENO1.glm.logistic.hybrid.adjusted', delim_whitespace=True)
    bonf_tokeep = bonf[['ID','BONF']]
    merged_2 = pd.merge(merged, bonf_tokeep, on='ID', how='right')

    #Print dimensions of the merged dataframe (just adding more columns)
    print(f'Merged dataframe shape: {merged_2.shape}')
    
    ## Save to CSV
    merged_2.to_csv(f'{WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}.allvariants_glm.txt', sep = '\t', index=False)

In [ ]:
#To run case-control analysis for exonic variants with covariates
ancestries = ['AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR','FIN', 'MDE', 'SAS', 'CAH']

for ancestry in ancestries:
    
    ! /home/jupyter/plink2 \
    --pfile {WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG \
    --keep {WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}.samplestokeep \
    --extract range {WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG.exonic.variantstoKeep.txt \
    --glm \
    --extract-if-info 'R2>=0.8' \
    --ci 0.95 \
    --adjust \
    --maf 0.01 \
    --mac 2 \
    --hwe 1e-5 0.001 \
    --covar {WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_covariate_file.txt \
    --covar-name SEX,AGE,PC1,PC2,PC3,PC4,PC5 \
    --covar-variance-standardize \
    --out {WORK_DIR}/POLG_GP2_NBA_R11/{ancestry}/{ancestry}_POLG.exonic